# 🪄 Chạy OpenAvatarChat miễn phí trên Colab GPU

Notebook này tự động:
1. Cài `uv` + Python 3.11 (đúng yêu cầu của dự án) và clone `OpenAvatarChat` (kèm **toàn bộ submodule**, đệ quy — bao gồm cả phần front-end mới nếu repo tách ra submodule riêng).
2. Cài dependencies (torch, fastapi, các model handler...).
3. Vá config: điền LLM/giọng đọc bạn chọn, tắt SSL tự ký (để cloudflared xử lý HTTPS ở tầng ngoài), và tuỳ theo **AVATAR_MODE** bạn chọn ở Bước 1:
   - `native_video` (mặc định, **khuyến nghị nếu máy có GPU khá**) — GIỮ NGUYÊN `RtcClient` (WebRTC) + bật `LiteAvatar` (mặt 2D). Server tự phục vụ đúng giao diện web gốc của OpenAvatarChat.
   - `lam_3d` — dùng bộ config `chat_with_lam.yaml`, bật `LamClient` + khuôn mặt 3D Gaussian Splatting của **LAM** (chính là công nghệ trong LAM_Audio2Expression bạn nhắc tới), render ngay trên trình duyệt. Nhẹ hơn cho server (chỉ chạy VAD+ASR cục bộ) nhưng cần tải thêm model LAM_audio2exp + wav2vec2.
   - `text_only_legacy` — đổi `RtcClient` → `WsClient`, tắt avatar để nhẹ hơn. Chỉ chữ + âm thanh qua WebSocket tự viết, KHÔNG có mặt avatar. Dùng cho panel "chế độ nhẹ" trong React.
4. **Vá ASR sang Groq Whisper** — model ASR mặc định (SenseVoiceSmall) chỉ hiểu Quan Thoại/Quảng Đông/Anh/Nhật/Hàn, KHÔNG hiểu tiếng Việt (nói tiếng Việt sẽ ra chữ Hán vô nghĩa). Notebook tự vá sang gọi Groq Whisper API (dùng chung key Groq ở Bước 1) để nhận diện tiếng Việt đúng.
5. Tải model ASR/TTS/LLM/Avatar cần thiết (LiteAvatar cho `native_video`, hoặc wav2vec2 + LAM_audio2exp cho `lam_3d`).
6. **Dọn sạch server cũ** còn sót lại từ lần chạy trước (nếu có) để tránh tình trạng tunnel vô tình trỏ vào server cũ chưa được vá.
7. Chạy server ở cổng `8282` và mở **Cloudflare Quick Tunnel** miễn phí để lấy 1 URL `https://...` public — dán **nguyên URL đó, không thêm path gì phía sau** vào panel tương ứng trong React.

> ⚠️ **Lưu ý quan trọng**: đây là môi trường Colab miễn phí — phiên sẽ tự ngắt sau vài giờ hoặc khi idle, và URL tunnel sẽ đổi mỗi lần chạy lại. Chỉ phù hợp để **test/demo**, không dùng cho sản phẩm chạy 24/7. GPU T4 miễn phí có thể chạy chậm hơn GPU cao cấp.
>
> ⚠️ **Về front-end của OpenAvatarChat**: từ bản 0.6.0 (04/2026), dự án tách front-end/back-end — giao diện Gradio cũ đã bị khai tử, thay bằng WebUI mới (Vue) được server phục vụ ở đường dẫn `/ui` (trang gốc `/` tự chuyển hướng sang đó). Nếu bạn từng quen vào thêm `/gradio/` ở cuối URL — **đừng làm vậy nữa**, server sẽ chỉ báo "Gradio page is no longer available". Cứ dán đúng URL gốc từ Bước 7.
>
> ⚠️ **Nếu chọn `native_video` hoặc `lam_3d`**: cần **TURN server** để xuyên NAT khi bạn và Colab VM không cùng mạng (99% lỗi "kẹt ở Connecting..." là do thiếu TURN — Cloudflare Quick Tunnel chỉ forward HTTP/WebSocket, không relay media UDP của WebRTC). Cách nhanh nhất, miễn phí: tạo tài khoản dùng thử tại https://www.twilio.com/, lấy **Account SID** + **Auth Token** ở Console, điền vào ô TWILIO_ACCOUNT_SID / TWILIO_AUTH_TOKEN ở Bước 1 — notebook tự lấy TURN credentials từ đó. Nếu để trống, vẫn chạy được nhưng chỉ ổn định khi test cùng mạng LAN với Colab (hiếm khi đúng, vì Colab luôn ở xa).
>
> 🔐 **Về API key**: notebook KHÔNG còn điền sẵn key mẫu nào — tự dán key Groq/Twilio của riêng bạn vào Bước 1 mỗi lần chạy (hoặc dùng Colab Secrets). Đừng lưu notebook đã điền key thật rồi chia sẻ cho người khác — nếu lỡ làm vậy, thu hồi/tạo lại key đó ngay tại nơi cấp key.
>
> ⚠️ **Mỗi khi chạy lại từ Bước 2** (phiên Colab mới, hoặc bấm "Run all" lại), các bước vá (4, 4b, 4c, 4d, 4e) đều cần chạy lại theo đúng thứ tự từ trên xuống — vì Bước 2 xoá và clone lại repo sạch, xoá luôn các bản vá cũ.

**Trước khi chạy**: vào `Runtime → Change runtime type → GPU (T4)` để bật GPU miễn phí.


In [ ]:
#@title Bước 0 — Kiểm tra GPU
!nvidia-smi || echo '⚠️ Chưa bật GPU. Vào Runtime > Change runtime type > GPU rồi chạy lại.'

In [ ]:
#@title Bước 1 — Cấu hình { display-mode: "form" }
GITHUB_REPO_URL = "https://github.com/HumanAIGC-Engineering/OpenAvatarChat.git" #@param {type:"string"}

#@markdown **LLM (bắt buộc)** — dùng API tương thích OpenAI. Mặc định đã đổi sang **Groq (miễn phí, không cần thẻ)**.
#@markdown Lấy API key miễn phí tại https://console.groq.com/keys rồi dán vào ô LLM_API_KEY bên dưới.
#@markdown ⚠️ Dán key CỦA BẠN mỗi lần chạy — đừng lưu lại notebook đã điền key thật rồi chia sẻ cho người khác.
LLM_API_URL = "https://api.groq.com/openai/v1" #@param {type:"string"}
LLM_API_KEY = "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" #@param {type:"string"}
LLM_MODEL_NAME = "openai/gpt-oss-20b" #@param {type:"string"}
SYSTEM_PROMPT = "Bạn là một trợ lý AI, trả lời ngắn gọn 2-3 câu bằng tiếng Việt." #@param {type:"string"}

#@markdown **Giọng đọc TTS (chỉ áp dụng cho `native_video`/`text_only_legacy`, dùng Edge TTS)** — ví dụ giọng nữ Việt: `vi-VN-HoaiMyNeural`, giọng nam: `vi-VN-NamMinhNeural`.
TTS_VOICE = "vi-VN-HoaiMyNeural" #@param {type:"string"}

#@markdown **Chế độ Avatar**
#@markdown - `native_video` (khuyến nghị nếu chỉ cần mặt 2D nhẹ): GIỮ NGUYÊN `RtcClient` (WebRTC) + bật `LiteAvatar`. Server tự phục vụ giao diện web gốc (WebUI mới, ở path `/ui`), nơi khuôn mặt avatar THẬT SỰ được vẽ ra. Dán URL vào panel "Video avatar thật" trong React. Cần cấu hình TURN bên dưới để hoạt động ổn định qua Internet.
#@markdown - `lam_3d`: dùng config `chat_with_lam.yaml`, bật `LamClient` — khuôn mặt 3D Gaussian Splatting của dự án LAM (aigc3d/LAM_Audio2Expression), render trên trình duyệt. Cần tải thêm model ở Bước 5 (wav2vec2 + LAM_audio2exp) và chọn hình mẫu ở LAM_ASSET bên dưới.
#@markdown - `text_only_legacy`: đổi sang `WsClient`, tắt avatar để tiết kiệm VRAM. Chỉ chữ + âm thanh, KHÔNG có mặt — dùng cho panel "chế độ nhẹ" trong React, không cần TURN.
AVATAR_MODE = "native_video" #@param ["native_video", "lam_3d", "text_only_legacy"]

#@markdown **Hình mẫu LAM (chỉ áp dụng khi AVATAR_MODE = `lam_3d`)** — 4 asset mẫu có sẵn trong repo (`src/handlers/client/h5_rendering_client/lam_samples/`).
LAM_ASSET = "barbara.zip" #@param ["barbara.zip", "yongen.zip", "sun.zip"]

#@markdown **TURN server (áp dụng cho `native_video` và `lam_3d`)** — WebRTC cần TURN để xuyên NAT khi client và Colab VM không cùng mạng. Cách dễ nhất: tài khoản dùng thử Twilio (miễn phí) tại https://www.twilio.com/ → Console → lấy Account SID + Auth Token.
TURN_PROVIDER = "twilio" #@param ["twilio", "custom", "none"]
TWILIO_ACCOUNT_SID = "ACc8c4f38d5b7276e1f6d0a253f2c985d7" #@param {type:"string"}
TWILIO_AUTH_TOKEN = "33652b20c71b11d2827bcfa38b4bfb14" #@param {type:"string"}
#@markdown Chỉ điền nếu chọn TURN_PROVIDER = "custom" (ví dụ tự chạy coturn):
CUSTOM_TURN_URL = "" #@param {type:"string"}
CUSTOM_TURN_USERNAME = "" #@param {type:"string"}
CUSTOM_TURN_CREDENTIAL = "" #@param {type:"string"}

if not LLM_API_KEY:
    print('⚠️ Chưa điền LLM_API_KEY. Vào https://console.groq.com/keys tạo key miễn phí, dán vào ô trên rồi chạy lại ô này.')
else:
    print('Đã lưu cấu hình — dùng Groq:', LLM_API_URL, '| model:', LLM_MODEL_NAME, '| avatar mode:', AVATAR_MODE)

if AVATAR_MODE in ("native_video", "lam_3d") and TURN_PROVIDER == "twilio" and not (TWILIO_ACCOUNT_SID and TWILIO_AUTH_TOKEN):
    print(f'⚠️ AVATAR_MODE={AVATAR_MODE} nhưng chưa điền Twilio Account SID/Auth Token — WebRTC dễ bị kẹt ở "Connecting..." nếu bạn và Colab không cùng mạng. Xem ghi chú ở ô markdown phía trên để lấy key miễn phí.')


In [ ]:
#@title Bước 2 — Clone repo + cài `uv`
%cd /content
!rm -rf OpenAvatarChat
!git clone --depth 1 {GITHUB_REPO_URL} OpenAvatarChat
%cd /content/OpenAvatarChat

# Từ bản 0.6.0 repo tách front-end/back-end và có thêm submodule; init đệ quy hết
# (thay vì chỉ init mỗi silero_vad như trước) để tránh thiếu file front-end/model
# dẫn tới lỗi khó hiểu ở các bước sau. --depth 1 vẫn giữ để tải nhanh.
!git submodule update --init --recursive --depth 1

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version


In [ ]:
#@title Bước 3 — Tạo venv Python 3.11 và cài dependencies (mất vài phút vì tải torch/CUDA)
!uv venv --python 3.11.7 .venv
!uv pip install --python .venv/bin/python --editable .

# QUAN TRỌNG: 'kích hoạt' venv cho cả session Colab (kể cả các lệnh gọi uv
# ngầm bên trong install.py ở Bước 5) — nếu không, uv sẽ tự rơi về Python
# hệ thống của Colab thay vì .venv/bin/python, gây lệch môi trường khi chạy demo.py.
import os
venv_path = os.path.abspath(".venv")
os.environ["VIRTUAL_ENV"] = venv_path
os.environ["PATH"] = os.path.join(venv_path, "bin") + ":" + os.environ["PATH"]
print("VIRTUAL_ENV =", os.environ["VIRTUAL_ENV"])

In [ ]:
#@title Bước 4 — Vá config: chọn chế độ avatar, tắt SSL tự ký, điền LLM/giọng đọc, cấu hình TURN
import pathlib

turn_yaml = None  # dùng chung cho mọi nhánh AVATAR_MODE, Bước 7 sẽ đọc biến này

if AVATAR_MODE == "lam_3d":
    # chat_with_lam.yaml có cấu trúc khác hẳn bản LiteAvatar: không có khối
    # RtcClient/WsClient/LiteAvatar mà thay bằng LamClient (render 3D Gaussian
    # Splatting ngay trên trình duyệt). Thay vì đoán chính xác từng chuỗi mặc định
    # như bản vá text-replace của các nhánh khác, ta parse YAML thành dict và tự dò
    # đúng khối theo "dấu hiệu" của từng loại handler — vẫn chạy đúng dù đội
    # OpenAvatarChat đổi model/API mặc định bên trong file.
    import yaml

    src_path = pathlib.Path("config/chat_with_lam.yaml")
    if not src_path.exists():
        raise SystemExit(
            "Không tìm thấy config/chat_with_lam.yaml — có thể repo gốc đã đổi tên/vị trí file.\n"
            "Mở thư mục config/ trên GitHub OpenAvatarChat, tìm file cấu hình dùng LamClient,\n"
            "rồi sửa lại đường dẫn src_path ở ô này cho khớp."
        )

    config = yaml.safe_load(src_path.read_text(encoding="utf-8"))

    def _walk(node):
        if isinstance(node, dict):
            yield node
            for v in node.values():
                yield from _walk(v)
        elif isinstance(node, list):
            for v in node:
                yield from _walk(v)

    found_llm = found_asset = False
    lam_client_block = None
    for block in _walk(config):
        if isinstance(block, dict) and "model_name" in block and "api_url" in block:
            block["model_name"] = LLM_MODEL_NAME
            block["api_url"] = LLM_API_URL
            if LLM_API_KEY:
                block["api_key"] = LLM_API_KEY
            if "system_prompt" in block:
                block["system_prompt"] = SYSTEM_PROMPT
            found_llm = True
        if isinstance(block, dict) and "asset_path" in block:
            block["asset_path"] = f"lam_samples/{LAM_ASSET}"
            found_asset = True
            lam_client_block = block  # giữ tham chiếu để gắn turn_config lồng vào đúng khối này
        if isinstance(block, dict) and "cert_file" in block:
            block["cert_file"] = ""
        if isinstance(block, dict) and "cert_key" in block:
            block["cert_key"] = ""

    if not found_llm:
        print("⚠️ Không tìm thấy khối LLM (model_name + api_url) trong chat_with_lam.yaml — "
              "mở file lên kiểm tra thủ công và điền LLM_API_KEY/LLM_API_URL/LLM_MODEL_NAME vào đúng chỗ.")
    if not found_asset:
        print("⚠️ Không tìm thấy khối LamClient (asset_path) — kiểm tra thủ công cấu trúc file, "
              "có thể repo đã đổi tên field.")

    if LLM_API_KEY:
        os.environ["DASHSCOPE_API_KEY"] = LLM_API_KEY

    if TURN_PROVIDER == "twilio" and TWILIO_ACCOUNT_SID and TWILIO_AUTH_TOKEN:
        turn_yaml = {"turn_provider": "twilio", "account_sid": TWILIO_ACCOUNT_SID, "auth_token": TWILIO_AUTH_TOKEN}
    elif TURN_PROVIDER == "custom" and CUSTOM_TURN_URL:
        turn_yaml = {"turn_provider": "turn_server", "urls": [CUSTOM_TURN_URL], "username": CUSTOM_TURN_USERNAME, "credential": CUSTOM_TURN_CREDENTIAL}
    if turn_yaml and lam_client_block is not None:
        # Cũng như RtcClient, turn_config phải lồng bên trong khối handler client
        # (ở đây là LamClient), không phải một khối "service:" riêng ở ngoài.
        lam_client_block["turn_config"] = turn_yaml
        print("Đã thêm cấu hình TURN (turn_config) lồng trong khối 'LamClient:'.")
    elif turn_yaml and lam_client_block is None:
        print("⚠️ Có cấu hình TURN nhưng không tìm thấy khối LamClient (asset_path) để lồng vào — "
              "kiểm tra thủ công cấu trúc chat_with_lam.yaml.")
        print("Đã thêm cấu hình TURN.")
    elif TURN_PROVIDER == "none":
        print("⚠️ Chưa cấu hình TURN cho lam_3d — nếu bạn và Colab không cùng mạng, kết nối có thể không ổn định. "
              "Chưa xác nhận 100% LamClient có bắt buộc TURN giống RtcClient hay không — nếu gặp lỗi kết nối, thử điền TURN ở Bước 1.")

    pathlib.Path("config/colab_config.yaml").write_text(
        yaml.dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8"
    )
    print(f"\nAVATAR_MODE=lam_3d: đã tạo config/colab_config.yaml từ chat_with_lam.yaml (asset: {LAM_ASSET})")

else:
    src_path = pathlib.Path("config/chat_with_openai_compatible_edge_tts.yaml")
    text = src_path.read_text(encoding="utf-8")

    rtc_block = (
        "      RtcClient:\n"
        "        module: client/rtc_client/client_handler_rtc\n"
        "        # max time a session will last for\n"
        "        connection_ttl: 900\n"
    )
    ws_block = (
        "      WsClient:\n"
        "        module: client/ws_client/ws_client_handler\n"
        "        connection_ttl: 900\n"
    )

    if rtc_block not in text:
        raise SystemExit(
            "Không tìm thấy khối 'RtcClient' đúng như dự kiến — có thể repo gốc đã đổi cấu trúc.\n"
            "Mở config/chat_with_openai_compatible_edge_tts.yaml và kiểm tra thủ công."
        )

    if AVATAR_MODE == "text_only_legacy":
        # Chế độ cũ: bỏ WebRTC hẳn, dùng client WebSocket tự viết trong React (panel "chế độ
        # nhẹ" — chỉ chữ + âm thanh, KHÔNG vẽ mặt) để né vấn đề TURN/NAT của WebRTC qua tunnel.
        text = text.replace(rtc_block, ws_block)
        text = text.replace(
            "      LiteAvatar:\n        module: avatar/liteavatar/avatar_handler_liteavatar",
            "      LiteAvatar:\n        enabled: False\n        module: avatar/liteavatar/avatar_handler_liteavatar",
        )
        print("AVATAR_MODE=text_only_legacy: đã đổi RtcClient → WsClient và tắt LiteAvatar (không có mặt avatar).")
    else:
        # native_video: GIỮ NGUYÊN RtcClient (WebRTC) — server sẽ tự phục vụ giao diện có mặt avatar thật.
        print("AVATAR_MODE=native_video: giữ nguyên RtcClient (WebRTC) + LiteAvatar — server sẽ tự phục vụ giao diện có mặt avatar thật.")

    # Tắt SSL tự ký để cloudflared xử lý HTTPS ở tầng ngoài (server chạy plain http nội bộ)
    text = text.replace('cert_file: "ssl_certs/localhost.crt"', 'cert_file: ""')
    text = text.replace('cert_key: "ssl_certs/localhost.key"', 'cert_key: ""')

    text = text.replace('model_name: "qwen-plus"', f'model_name: "{LLM_MODEL_NAME}"')
    text = text.replace(
        'api_url: "https://dashscope.aliyuncs.com/compatible-mode/v1"',
        f'api_url: "{LLM_API_URL}"',
    )
    text = text.replace(
        'system_prompt: "\u8bf7\u4f60\u626e\u6f14\u4e00\u4e2a AI \u52a9\u624b\uff0c\u7528\u7b80\u77ed\u7684\u4e24\u4e09\u53e5\u5bf9\u8bdd\u6765\u56de\u7b54\u7528\u6237\u7684\u95ee\u9898\uff0c\u5e76\u5728\u5bf9\u8bdd\u5185\u5bb9\u4e2d\u52a0\u5165\u5408\u9002\u7684\u6807\u70b9\u7b26\u53f7\uff0c\u4e0d\u9700\u8981\u8ba8\u8bba\u6807\u70b9\u7b26\u53f7\u76f8\u5173\u7684\u5185\u5bb9"',
        f'system_prompt: "{SYSTEM_PROMPT}"',
    )
    text = text.replace('voice: "zh-CN-XiaoxiaoNeural"', f'voice: "{TTS_VOICE}"')

    if LLM_API_KEY:
        os.environ["DASHSCOPE_API_KEY"] = LLM_API_KEY
        text = text.replace(
            "        api_url:",
            f'        api_key: "{LLM_API_KEY}"\n        api_url:',
        )

    if AVATAR_MODE == "native_video" and TURN_PROVIDER == "twilio" and TWILIO_ACCOUNT_SID and TWILIO_AUTH_TOKEN:
        turn_yaml = (
            "        turn_config:\n"
            "          turn_provider: \"twilio\"\n"
            f"          account_sid: \"{TWILIO_ACCOUNT_SID}\"\n"
            f"          auth_token: \"{TWILIO_AUTH_TOKEN}\"\n"
        )
    elif AVATAR_MODE == "native_video" and TURN_PROVIDER == "custom" and CUSTOM_TURN_URL:
        turn_yaml = (
            "        turn_config:\n"
            "          turn_provider: \"turn_server\"\n"
            f"          urls: [\"{CUSTOM_TURN_URL}\"]\n"
            f"          username: \"{CUSTOM_TURN_USERNAME}\"\n"
            f"          credential: \"{CUSTOM_TURN_CREDENTIAL}\"\n"
        )

    if turn_yaml:
        # QUAN TRỌNG: field đúng tên là 'turn_config' và nó phải LỒNG BÊN TRONG
        # khối 'RtcClient:' (ngang hàng với module/connection_ttl) — KHÔNG phải
        # một khối 'service: rtc_config:' riêng ở ngoài như bản vá trước. Bằng
        # chứng từ chính log khởi động server (Bước 7): dòng đăng ký handler in ra
        # 'Registered handler RtcClient(...) ... turn_config=None ...' — nghĩa là
        # turn_config là một field thuộc config của chính RtcClient. Chèn ngay sau
        # connection_ttl trong rtc_block, cùng cấp thụt lề (8 dấu cách).
        text = text.replace(rtc_block, rtc_block + turn_yaml)
        print("Đã thêm cấu hình TURN (turn_config) lồng đúng chỗ trong khối 'RtcClient:'.")
    elif AVATAR_MODE == "native_video":
        print("⚠️ Chưa cấu hình TURN (TURN_PROVIDER='none' hoặc thiếu Account SID/Auth Token/URL). "
              "Nếu bạn và Colab không cùng mạng — gần như luôn đúng — WebRTC rất có thể bị kẹt ở "
              "'Connecting...'. Quay lại Bước 1 để điền Twilio, rồi chạy lại từ ô này.")
    elif AVATAR_MODE == "native_video":
        print("⚠️ Chưa cấu hình TURN (TURN_PROVIDER='none' hoặc thiếu Account SID/Auth Token/URL). "
              "Nếu bạn và Colab không cùng mạng — gần như luôn đúng — WebRTC rất có thể bị kẹt ở "
              "'Connecting...'. Quay lại Bước 1 để điền Twilio, rồi chạy lại từ ô này.")

    pathlib.Path("config/colab_config.yaml").write_text(text, encoding="utf-8")
    print("\nĐã tạo config/colab_config.yaml:\n")
    print(text)


In [ ]:
#@title Bước 4b — Vá install.py để ép dùng đúng .venv (khắc phục uv tự rơi về Python hệ thống)
import pathlib

install_path = pathlib.Path("install.py")
install_text = install_path.read_text(encoding="utf-8")

target = '["uv", "pip", "install"'
patched = '["uv", "pip", "install", "--python", ".venv/bin/python"'

count = install_text.count(target)
if count == 0:
    print("⚠️ Không tìm thấy chuỗi cần vá — có thể install.py đã đổi cấu trúc, kiểm tra thủ công.")
else:
    install_text = install_text.replace(target, patched)
    install_path.write_text(install_text, encoding="utf-8")
    print(f"Đã vá {count} lệnh 'uv pip install' bên trong install.py để luôn chỉ định --python .venv/bin/python")

In [ ]:
#@title Bước 4c — Vá ASR: SenseVoice (không hiểu tiếng Việt) → Groq Whisper (hiểu tiếng Việt)
import pathlib

asr_path = pathlib.Path("src/handlers/asr/sensevoice/asr_handler_sensevoice.py")
asr_text = asr_path.read_text(encoding="utf-8")

# Đây là dòng thật trong source SenseVoice handler của OpenAvatarChat — nơi nó gọi
# model FunASR cục bộ để nhận diện giọng nói. Model này (iic/SenseVoiceSmall) CHỈ
# hiểu Quan Thoại / Quảng Đông / Anh / Nhật / Hàn — không có tiếng Việt, nên khi bạn
# nói tiếng Việt nó ép âm thanh vào âm tiết Quảng Đông gần giống nhất -> ra chữ Hán
# vô nghĩa. Ta thay đúng dòng này bằng 1 lệnh gọi Groq Whisper (hiểu tiếng Việt tốt),
# và giữ nguyên mọi thứ khác (VAD, buffer, threading) vì phần đó đã chạy đúng.
anchor = "res = self.model.generate(input=output_audio, batch_size_s=10)"

if anchor not in asr_text:
    raise SystemExit(
        "Không tìm thấy dòng gọi model SenseVoice như dự kiến — có thể repo gốc đã đổi "
        "cấu trúc file src/handlers/asr/sensevoice/asr_handler_sensevoice.py.\n"
        "Mở file đó lên, tìm dòng có 'self.model.generate(' và báo lại nội dung dòng đó "
        "để chỉnh lại patch cho khớp."
    )

# Hàm gọi Groq Whisper, trả về đúng định dạng [{'text': ...}] mà code gốc bên dưới
# đang mong đợi (dòng kế tiếp re.sub(r"<\|.*?\|>", "", res[0]['text']) vẫn chạy bình
# thường vì text thường không chứa các tag đó).
groq_whisper_helper = '''
def _groq_whisper_transcribe(audio_data, sample_rate=16000, model="whisper-large-v3-turbo", language="vi"):
    """Transcribe qua Groq Whisper API — thay cho SenseVoice cục bộ để hỗ trợ tiếng Việt."""
    import requests, numpy as np, io, wave, os as _os
    api_key = _os.environ.get("GROQ_API_KEY") or _os.environ.get("DASHSCOPE_API_KEY")
    if not api_key:
        print("[groq_whisper] Thiếu GROQ_API_KEY trong biến môi trường.")
        return [{"text": ""}]
    try:
        arr = np.asarray(audio_data)
        if arr.dtype != np.int16:
            arr = np.clip(arr, -1.0, 1.0)
            arr = (arr * 32767.0).astype(np.int16)
        buf = io.BytesIO()
        with wave.open(buf, "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sample_rate)
            wf.writeframes(arr.tobytes())
        buf.seek(0)
        resp = requests.post(
            "https://api.groq.com/openai/v1/audio/transcriptions",
            headers={"Authorization": f"Bearer {api_key}"},
            files={"file": ("audio.wav", buf, "audio/wav")},
            data={"model": model, "language": language, "response_format": "json"},
            timeout=30,
        )
        resp.raise_for_status()
        text = resp.json().get("text", "").strip()
        print(f"[groq_whisper] transcript: {text!r}")
        return [{"text": text}]
    except Exception as e:
        print(f"[groq_whisper] transcribe error: {e}")
        return [{"text": ""}]


'''

# Chèn hàm helper vào ngay đầu file (an toàn — không phụ thuộc import sẵn có của file).
if "_groq_whisper_transcribe" not in asr_text:
    asr_text = groq_whisper_helper + asr_text

# Thay dòng gọi SenseVoice cục bộ bằng lệnh gọi Groq Whisper.
asr_text = asr_text.replace(anchor, "res = _groq_whisper_transcribe(output_audio, sample_rate=16000)")

asr_path.write_text(asr_text, encoding="utf-8")

# Groq Whisper dùng chung key với LLM đã điền ở Bước 1.
import os
if LLM_API_KEY:
    os.environ["GROQ_API_KEY"] = LLM_API_KEY

print("✅ Đã vá ASR handler: SenseVoice → Groq Whisper (language=\'vi\').")
print("   (Model SenseVoice vẫn được tải ở Bước 5 nhưng không còn được dùng để nhận diện — vô hại, chỉ tốn thêm ít thời gian tải.)")
print("   ⚠️ Nhắc nhớ: nếu bạn chạy lại Bước 2 (clone lại repo) ở 1 phiên sau, ô patch này PHẢI được chạy lại trước Bước 5/7.")


In [ ]:
#@title Bước 4d — Vá VAD: tránh bị AI cắt ngang khi đang nói giữa chừng
import pathlib

vad_path = pathlib.Path("config/colab_config.yaml")
vad_text = vad_path.read_text(encoding="utf-8")

# Mặc định của OpenAvatarChat: end_delay=5000 samples ở 16kHz ~= chỉ 312ms im lặng
# là VAD đã chốt "bạn nói xong" và đẩy câu (có thể còn dang dở) sang ASR/LLM ngay.
# Một khoảng ngừng tự nhiên giữa câu (hít thở, ngập ngừng) thường dài hơn thế,
# nên VAD hay cắt ngang. Ta tăng các ngưỡng liên quan lên để chịu được khoảng
# ngừng tự nhiên (~1s) mà vẫn phản hồi nhanh khi bạn thực sự dứt lời.
old_vad_block = (
    "      SileroVad:\n"
    "        module: vad/silerovad/vad_handler_silero\n"
    "        speaking_threshold: 0.5\n"
    "        start_delay: 2048\n"
    "        end_delay: 5000\n"
    "        buffer_look_back: 5000\n"
    "        speech_padding: 512\n"
)
new_vad_block = (
    "      SileroVad:\n"
    "        module: vad/silerovad/vad_handler_silero\n"
    "        speaking_threshold: 0.5\n"
    "        start_delay: 2048\n"
    "        end_delay: 16000\n"
    "        early_end_delay: 6000\n"
    "        early_end_repeat_delay: 4800\n"
    "        buffer_look_back: 5000\n"
    "        speech_padding: 512\n"
)

if old_vad_block not in vad_text:
    raise SystemExit(
        "Không tìm thấy khối 'SileroVad' đúng như dự kiến trong config/colab_config.yaml "
        "— có thể Bước 4 chưa chạy, hoặc repo gốc đã đổi cấu trúc.\n"
        "Mở file đó lên, tìm khối 'SileroVad:' và tự thêm/sửa thủ công 3 dòng:\n"
        "  end_delay: 16000\n"
        "  early_end_delay: 6000\n"
        "  early_end_repeat_delay: 4800"
    )

vad_text = vad_text.replace(old_vad_block, new_vad_block)
vad_path.write_text(vad_text, encoding="utf-8")
print("Đã tăng ngưỡng VAD trong config/colab_config.yaml:\n")
print(new_vad_block)
print(
    "Ghi chú: nếu vẫn thấy bị cắt ngang, có thể tăng end_delay lên 20000-24000 "
    "(1.25-1.5s). Nếu ngược lại thấy AI phản hồi chậm rõ rệt sau khi bạn dứt lời, "
    "giảm bớt xuống ~12000."
)


In [ ]:
#@title Bước 4e — Vá TTS: giữ dấu tiếng Việt khi đọc (Edge TTS)
import pathlib

tts_path = pathlib.Path("src/handlers/tts/edgetts/tts_handler_edgetts.py")
tts_text = tts_path.read_text(encoding="utf-8")

# Handler TTS gốc được viết cho tiếng Trung: filter_text() chỉ giữ lại chữ Latin
# không dấu, số, và chữ Hán (\u4e00-\u9fff) trước khi đưa văn bản cho Edge TTS đọc.
# Mọi ký tự có dấu tiếng Việt (ạ, ộ, ế, ữ, đ...) đều bị regex này xoá sạch, nên dù
# giọng đọc là vi-VN-HoaiMyNeural (nữ, tiếng Việt) vẫn nghe như đọc sai/lạ vì mất
# hết dấu và thanh điệu. Ta thêm dải Unicode tiếng Việt vào danh sách ký tự được giữ.
old_pattern = (
    '        pattern = r"[^a-zA-Z0-9\\u4e00-\\u9fff,.\\~!?，。！？ ]"  '
    '# 匹配不在范围内的字符\n'
)
new_pattern = (
    '        pattern = r"[^a-zA-Z0-9\\u00C0-\\u1EF9\\u4e00-\\u9fff,.\\~!?，。！？ ]"  '
    '# 匹配不在范围内的字符 (đã thêm \\u00C0-\\u1EF9 để giữ dấu tiếng Việt)\n'
)

if old_pattern not in tts_text:
    raise SystemExit(
        "Không tìm thấy dòng 'pattern = ...' đúng như dự kiến trong "
        "src/handlers/tts/edgetts/tts_handler_edgetts.py — có thể repo gốc đã đổi.\n"
        "Mở file đó, tìm hàm filter_text(), và tự thêm \\u00C0-\\u1EF9 vào regex "
        "(ngay trước \\u4e00-\\u9fff) để không bị xoá dấu tiếng Việt."
    )

tts_text = tts_text.replace(old_pattern, new_pattern)
tts_path.write_text(tts_text, encoding="utf-8")
print("Đã vá filter_text() trong tts_handler_edgetts.py — giờ dấu tiếng Việt sẽ được giữ lại.")
print("Lưu ý: chạy cell này SAU Bước 2 (đã clone repo) và TRƯỚC Bước 5 (tải model / chạy server).")


In [ ]:
#@title Bước 5 — Tải model (ASR, LLM handler phụ trợ, và khuôn mặt avatar tuỳ AVATAR_MODE)
!.venv/bin/python install.py --config config/colab_config.yaml

if AVATAR_MODE == "native_video":
    print("\nAVATAR_MODE=native_video -> tải thêm model khuôn mặt LiteAvatar (nếu install.py chưa tự tải)...")
    import subprocess
    result = subprocess.run(["bash", "scripts/download_liteavatar_weights.sh"], capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print("⚠️ Không chạy được scripts/download_liteavatar_weights.sh (có thể phiên bản repo hiện tại đã tự tải model này trong install.py, hoặc tên script đã đổi).")
        print(result.stderr[-1500:])
        print("Nếu Bước 7 báo lỗi thiếu model LiteAvatar, mở repo trên GitHub và tìm script tải model tương ứng trong thư mục scripts/.")

elif AVATAR_MODE == "lam_3d":
    print("\nAVATAR_MODE=lam_3d -> tải 2 model cần cho LAM: wav2vec2-base-960h + LAM_audio2exp...")
    import subprocess, pathlib
    pathlib.Path("models").mkdir(exist_ok=True)

    if not pathlib.Path("models/wav2vec2-base-960h").exists():
        r1 = subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://www.modelscope.cn/AI-ModelScope/wav2vec2-base-960h.git",
             "./models/wav2vec2-base-960h"],
            capture_output=True, text=True,
        )
        print(r1.stdout[-1500:])
        if r1.returncode != 0:
            print("⚠️ Không tải được wav2vec2-base-960h từ ModelScope. Log lỗi:\n", r1.stderr[-1500:])
    else:
        print("models/wav2vec2-base-960h đã tồn tại, bỏ qua.")

    if not pathlib.Path("models/LAM_audio2exp").exists() or not any(pathlib.Path("models/LAM_audio2exp").iterdir()):
        pathlib.Path("models/LAM_audio2exp").mkdir(parents=True, exist_ok=True)
        r2 = subprocess.run(
            ["wget", "-q",
             "https://huggingface.co/3DAIGC/LAM_audio2exp/resolve/main/LAM_audio2exp_streaming.tar",
             "-P", "models/LAM_audio2exp/"],
            capture_output=True, text=True,
        )
        if r2.returncode != 0:
            print("⚠️ Không tải được LAM_audio2exp_streaming.tar từ HuggingFace. Log lỗi:\n", r2.stderr[-1500:])
        else:
            r3 = subprocess.run(
                ["tar", "-xzvf", "models/LAM_audio2exp/LAM_audio2exp_streaming.tar", "-C", "models/LAM_audio2exp"],
                capture_output=True, text=True,
            )
            print(r3.stdout[-1000:])
            if r3.returncode != 0:
                print("⚠️ Giải nén LAM_audio2exp lỗi:\n", r3.stderr[-1000:])
    else:
        print("models/LAM_audio2exp đã có sẵn dữ liệu, bỏ qua.")

    print(
        "\n⚠️ Nếu các lệnh tải model trên báo lỗi (URL đổi chỗ, mirror chặn theo khu vực...), "
        "vào README chính của OpenAvatarChat trên GitHub, mục cài đặt cho LAM, để lấy lại đúng "
        "lệnh tải mới nhất — 2 model này đôi khi được host lại ở cả ModelScope lẫn HuggingFace, "
        "URL có thể đổi giữa các bản release."
    )


In [ ]:
#@title Bước 6 — Cài Cloudflare Tunnel (miễn phí, không cần tài khoản)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
#@title Bước 6b — Dọn sạch server cũ (nếu có) đang chiếm cổng 8282
import subprocess, time

# Kill mọi tiến trình python đang chạy src/demo.py từ (các) lần chạy trước còn sót lại
# trong VM Colab này. Nếu không dọn, Bước 7 bên dưới sẽ báo lỗi "address already in use"
# và server MỚI (đã có patch ASR) sẽ không khởi động được — trong khi tunnel Cloudflare
# vẫn vô tình trỏ vào server CŨ (chưa patch) khiến bạn tưởng patch không có tác dụng.
subprocess.run("pkill -9 -f 'src/demo.py' || true", shell=True)
subprocess.run("fuser -k 8282/tcp 2>/dev/null || true", shell=True)
time.sleep(3)

# Xác nhận cổng 8282 đã trống
check = subprocess.run("lsof -i:8282 || true", shell=True, capture_output=True, text=True)
if check.stdout.strip():
    print("⚠️ Cổng 8282 vẫn còn tiến trình đang giữ:\n" + check.stdout)
    print("   Nếu Bước 7 vẫn báo 'address already in use', vào Runtime → Restart session rồi chạy lại từ Bước 2.")
else:
    print("✅ Cổng 8282 đã trống — sẵn sàng khởi động server mới ở Bước 7.")


In [ ]:
#@title Bước 7 — Chạy server + mở tunnel, lấy URL public
import re
import socket
import subprocess
import time

server_log = open("/content/server.log", "w")
server_proc = subprocess.Popen(
    [".venv/bin/python", "src/demo.py", "--config", "config/colab_config.yaml"],
    stdout=server_log, stderr=subprocess.STDOUT, cwd="/content/OpenAvatarChat",
)
print(f"Server PID: {server_proc.pid} — đang khởi động, chờ tới khi cổng 8282 sẵn sàng (tối đa 5 phút)...")

def port_is_open(host, port, timeout=1.0):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

MAX_WAIT_SECONDS = 300
waited = 0
server_ready = False
while waited < MAX_WAIT_SECONDS:
    if server_proc.poll() is not None:
        print(f"❌ Server đã thoát sớm (exit code {server_proc.returncode}) — xem server.log bên dưới:")
        print(open("/content/server.log").read()[-4000:])
        break
    if port_is_open("localhost", 8282):
        server_ready = True
        print(f"✅ Server đã mở cổng 8282 sau ~{waited}s.")
        break
    time.sleep(3)
    waited += 3
    if waited % 15 == 0:
        print(f"  ... vẫn đang chờ ({waited}s) — model ASR/TTS/LLM/Avatar có thể cần thời gian tải/nạp GPU lần đầu.")

if not server_ready:
    if server_proc.poll() is None:
        print("⚠️ Hết thời gian chờ nhưng server vẫn đang chạy — có thể cần thêm thời gian. Xem server.log:")
        print(open("/content/server.log").read()[-4000:])
    raise SystemExit("Dừng lại: server chưa sẵn sàng, đừng mở tunnel để tránh nhầm lẫn URL chết.")

tunnel_log = open("/content/tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8282"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    log_text = open("/content/tunnel.log").read()
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("✅ Server + tunnel đã sẵn sàng!\n")
    if AVATAR_MODE in ("native_video", "lam_3d"):
        print(f"Dán ĐÚNG URL GỐC này (KHÔNG thêm /gradio/ hay bất kỳ path nào khác) vào ô 'Video avatar thật' trong panel My AI Avatar:\n\n   {public_url}\n")
        print("Từ bản 0.6.0, đây là WebUI mới của OpenAvatarChat, được phục vụ ở path '/ui' — truy cập URL gốc '/' sẽ tự chuyển hướng sang đó.")
        print("Nếu vào URL gốc mà KHÔNG tự chuyển hướng và không thấy giao diện, thử thêm '/ui/' vào cuối URL trước khi báo lỗi.")
        print("Lưu ý: trang sẽ tự hỏi quyền camera/micro. Nếu bị chặn khi nhúng trong iframe, dùng nút mở-tab-mới trong panel.")
        if TURN_PROVIDER == "none" or not turn_yaml:
            print("\n⚠️ Chưa có TURN — nếu 'Connecting...' không qua được, quay lại Bước 1 điền Twilio rồi chạy lại từ Bước 4.")
    else:
        ws_url = public_url.replace("https://", "wss://")
        print(f"Dán URL này vào ô 'Server URL' trong panel 'chế độ nhẹ' (chỉ chữ + âm thanh, không có mặt):\n\n   {ws_url}\n")
else:
    print("⚠️ Chưa lấy được URL tunnel. Kiểm tra log bên dưới:")
    print(open("/content/tunnel.log").read())
    print("\n--- server.log ---")
    print(open("/content/server.log").read()[-3000:])


In [ ]:
#@title (Tuỳ chọn) Xem log server / tunnel trực tiếp để debug
print("----- server.log (300 dòng cuối) -----")
!tail -n 300 /content/server.log
print("\n----- tunnel.log -----")
!cat /content/tunnel.log

In [ ]:
#@title (Tuỳ chọn) Dừng server + tunnel
try:
    server_proc.terminate()
    tunnel_proc.terminate()
    print("Đã dừng server và tunnel.")
except NameError:
    print("Chưa có process nào đang chạy trong session này.")

## Ghi chú
- Mỗi lần Colab ngắt phiên (đóng tab, hết giờ, idle quá lâu), bạn phải chạy lại **toàn bộ notebook từ Bước 2** và URL tunnel ở Bước 7 sẽ **thay đổi** — nhớ cập nhật lại panel React tương ứng.
- ASR mặc định của dự án gốc (SenseVoiceSmall) không hỗ trợ tiếng Việt — notebook này đã tự vá (Bước 4c) để dùng Groq Whisper thay thế, giúp nhận diện tiếng Việt chính xác. Nếu muốn quay lại SenseVoice gốc (ví dụ để test tiếng Trung/Anh/Nhật/Hàn), chỉ cần bỏ qua ô Bước 4c.
- **Về giao diện web ("/gradio/" báo lỗi)**: từ bản 0.6.0, OpenAvatarChat tách front-end/back-end; giao diện Gradio cũ đã bị khai tử và thay bằng WebUI mới (Vue), phục vụ ở path `/ui`. Nếu bạn vào `/gradio/` sẽ chỉ thấy thông báo dùng WebUI mới — chỉ cần dùng đúng URL gốc từ Bước 7 (không thêm path), server sẽ tự chuyển hướng.
- **Về khuôn mặt avatar — 2 lựa chọn thật sự khác nhau**:
  - `native_video` (mặc định): mặt 2D **LiteAvatar**, server render sẵn video rồi stream qua WebRTC.
  - `lam_3d`: mặt 3D **Gaussian Splatting** của LAM (đúng công nghệ trong LAM_Audio2Expression bạn nhắc tới) — dữ liệu biểu cảm được server gửi qua, còn việc dựng hình 3D diễn ra ngay trên trình duyệt (nhẹ hơn cho server, hỗ trợ nhiều phiên đồng thời hơn).
  Cả hai đều được vẽ bởi WebUI gốc của OpenAvatarChat (không phải client WebSocket tự viết trong React — xem comment đầu file `src/services/openAvatarChatClient.js`), nên panel "Video avatar thật" chỉ nhúng iframe URL gốc là đủ, không cần code thêm gì ở phía React.
- `native_video` và `lam_3d` đều cần **TURN** để hoạt động ổn định qua Internet (không cùng mạng với Colab) — điền Twilio Account SID/Auth Token miễn phí ở Bước 1, hoặc TURN server tự host (coturn) nếu có.
- 🔐 Notebook không còn điền sẵn API key mẫu nào. Nếu bạn từng lưu/chia sẻ một bản notebook đã điền key thật, hãy thu hồi/tạo lại key đó ngay tại nơi cấp key (Groq Console, Twilio Console...).
